LOADING AND EXPLORING OUR DATASET


In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import *

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [0]:
df = spark.table("us_emissions_catalog_8_5.raw.emissions_data")

In [0]:
display(spark.createDataFrame(pd.DataFrame(df.head(10))))

In [0]:
df.printSchema()

In [0]:
print(f"rows: {df.count()}")

In [0]:
print(len(df.columns))

In [0]:
print(df.count())

In [0]:
display(df.limit(100))

**_Let's dive into some statistics_**

In [0]:
df.select(F.sum("population")).show()

In [0]:
df.select(
    F.avg("population").alias("Average"),
    F.sum("population").alias("Total"),
    F.max("population").alias("Maximum"),
    F.min("population").alias("Minimum")
).show()

In [0]:
import re
safe_df = df.toDF(*[re.sub(r'[^a-zA-Z0-9_]', '_', c) for c in df.columns])
display(safe_df.describe())

In [0]:
pop_average = df.select(F.avg("population"))
display(pop_average.select(F.format_number("avg(population)", 3).alias("Average Population")))

In [0]:
display(df.select(F.max("population").alias("Maxium population")))

In [0]:
display(df.select(min("population").alias("minimum population")))

In [0]:
info_stats= df.select(
    F.count("*").alias("Number of rows"),
    F.lit(len(df.columns)).alias("Number of columns"),
    F.avg("population").alias("Average population"),
)
display(info_stats)

In [0]:
df.printSchema()

**Data transformation steps**

In [0]:
df.columns

In [0]:
df.select("state_id", "county_id").printSchema()

In [0]:
nuls_in =df.select(
    [F.count(F.when(F.col(f"`{c}`").isNull(), c)).alias(c) for c in df.columns])
display(nuls_in)

Databricks data profile. Run in Databricks to view.

In [0]:
display(df.select("consolidated_city-county").distinct())

In [0]:
display(
    df.groupBy("county_id")
      .count()
      .filter("count > 1")
)

In [0]:
df.select("county_id").distinct().count()

In [0]:
%sql
SELECT state_id, county_id, count(*) FROM us_emissions_catalog_8_5.raw.emissions_data 
GROUP BY state_id, county_id
HAVING count(*) > 1
ORDER BY count(*) DESC

# ## ### > Transformation **stage**

Most of the columns do not have the correct data types, so, we will proceed with first, making sure the all have the correct data types.... (make sure the columns names are properly changed first before runninng this part)

In [0]:
from pyspark.sql import functions as F

# 1. Rename columns
rename_map = {
    "county_state_name": "county_state_label",
    "consolidated_city-county": "consolidated_city_county",
    "latitude": "lati",
    "longitude": "long",
    "population_cohort": "pop_size_cohort",
    "employment_cohort": "emp_size_cohort",
    "doe_climate_zone": "climate_zone",
    "consumption (MWh)": "elec_consumption_mwh",
    "expenditures in Millions": "elec_expendit_millions",
    "consumption (MWh/capita)": "elec_consumption_mwh_capita",
    "utility customers": "elec_utility_customers",
    "GHG emissions mtons CO2e": "ghg_emissions_mtco2e",
    "consumption (TcF)": "gas_consumption_tcf",
    "consumption (TcF/capita)": "gas_consumption_tcf_capita",
    "occuped housing units": "occ_housing_units",
    "consumption (gallons)": "water_consumption_gal",
    "consumption (gallons/capita)": "water_consumption_gal_capita",
    "vehicle miles traveled (miles)": "vmt_total",
    "vehicle miles traveled (miles/capita)": "vmt_capita"
}
df2 = df.withColumnsRenamed(rename_map)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, LongType

numeric_cols = {
    # Integers
    "population": LongType(),
    "employment": LongType(),
    "elec_utility_customers": LongType(),
    "occ_housing_units": LongType(),

    # Decimals
    "lati": DoubleType(),
    "long": DoubleType(),
    "elec_consumption_mwh": DoubleType(),
    "elec_expendit_millions": DoubleType(),
    "elec_consumption_mwh_capita": DoubleType(),
    "ghg_emissions_mtco2e": DoubleType(),
    "gas_consumption_tcf": DoubleType(),
    "gas_consumption_tcf_capita": DoubleType(),
    "water_consumption_gal": DoubleType(),
    "water_consumption_gal_capita": DoubleType(),
    "vmt_total": DoubleType(),
    "vmt_capita": DoubleType(),
}

In [0]:
from pyspark.sql.types import LongType

for col_name, dtype in numeric_cols.items():
    type_sql = "BIGINT" if isinstance(dtype, LongType) else "DOUBLE"
    df2 = df2.withColumn(
        col_name,
        F.expr(f"try_cast(regexp_replace(`{col_name}`, ',', '') AS {type_sql})")
    )


In [0]:
# 2. Replace Y/null with Yes/No
df2 = df2.withColumn(
    "consolidated_city_county",
    F.when(F.col("consolidated_city_county") == "Y", "Yes")
     .otherwise("No")
)


In [0]:
# 3 drop fully null columns
df2 = df2.drop(
    "buildings",
    "area (sq. ft.)",
    "establishments"
)

In [0]:
display(df2)

In [0]:
df2.printSchema()

In [0]:
print(
    f"Rows1: {df.count()}, columns1: {len(df.columns)}\n"
    , f"Rows2: {df2.count()}, columns2: {len(df2.columns)}"
)

In [0]:
df2.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("us_emissions_catalog_8_5.raw.emission_df2")

In [0]:
display(
    spark.table("us_emissions_catalog_8_5.raw.emission_df2")
)

In [0]:
# Verify
df2.printSchema()